**Project Code:** P001-MVB
**Stage:** 05_MVB_Model-Training
**Notebook Version:** v1.0
**Author:** Abubakar Amidu
**Programme:** 3MTT DeepTech Cohort 2 — DS/ML Mentorship
**Last Updated:** 25 July 2026

---

# MVB-05 — Model Training

**Project:** P001-MVB — Explainable and Responsible AI for Differentiating Bacterial and Viral Meningitis: A Clinical Decision Support System

## Purpose

Train and compare Logistic Regression, Random Forest, and XGBoost on both Feature Set A (Full) and Feature Set B (Restricted), producing six model variants for evaluation in Stage 06.

## Inputs

- `MVB-04-results/MVB-04-feature-set-A-full.csv`
- `MVB-04-results/MVB-04-feature-set-B-restricted.csv`

## Outputs

- 6 trained models: `Models/` (Logistic Regression, Random Forest, XGBoost × Feature Sets A/B)
- Baseline metrics table: `MVB-05-results/MVB-05-baseline-metrics.csv`
- This notebook
- Updated documentation

In [ ]:

# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Set working directory to Stage 05 folder
import os
os.chdir('/content/drive/MyDrive/P001-MVB/05_MVB_Model-Training')
os.makedirs("Models", exist_ok=True)
os.makedirs("MVB-05-results", exist_ok=True)

In [ ]:

# Imports
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Reproducibility standard, consistent with Stages 03-04
np.random.seed(42)

In [ ]:

# Load both feature sets from Stage 04
df_a = pd.read_csv("../04_MVB_Feature-Engineering/MVB-04-results/MVB-04-feature-set-A-full.csv")
df_b = pd.read_csv("../04_MVB_Feature-Engineering/MVB-04-results/MVB-04-feature-set-B-restricted.csv")

print("Feature Set A shape:", df_a.shape)
print("Feature Set B shape:", df_b.shape)

Feature Set A shape: (1133, 11)
Feature Set B shape: (1133, 10)


In [ ]:

# Integrity check — confirm feature sets match Stage 04's documented output
assert df_a.shape == (1133, 11)
assert df_b.shape == (1133, 10)
assert "Pathogen_Present" in df_a.columns
assert "Pathogen_Present" not in df_b.columns
print("Integrity check passed: feature sets match Stage 04 documentation.")

Integrity check passed: feature sets match Stage 04 documentation.


## Train/Test Split

An 80/20 stratified split is used, preserving the class balance in both training and test sets. A fixed random seed (42) ensures reproducibility across both feature sets.

In [ ]:
# Build train/test splits for a given dataframe and feature list
def prepare_split(df, feature_cols):
    X = df[feature_cols]
    y = df["Diagnosis"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
    print(f"Train class balance:\n{y_train.value_counts(normalize=True).round(3)}")
    return X_train, X_test, y_train, y_test

# Feature columns for each set (target column excluded)
feature_set_a_cols = [c for c in df_a.columns if c != "Diagnosis"]
feature_set_b_cols = [c for c in df_b.columns if c != "Diagnosis"]

# Split Feature Set A
print("=== Feature Set A ===")
X_train_a, X_test_a, y_train_a, y_test_a = prepare_split(df_a, feature_set_a_cols)
print()

# Split Feature Set B
print("=== Feature Set B ===")
X_train_b, X_test_b, y_train_b, y_test_b = prepare_split(df_b, feature_set_b_cols)

=== Feature Set A ===
Train shape: (906, 10), Test shape: (227, 10)
Train class balance:
Diagnosis
1    0.525
0    0.475
Name: proportion, dtype: float64

=== Feature Set B ===
Train shape: (906, 9), Test shape: (227, 9)
Train class balance:
Diagnosis
1    0.525
0    0.475
Name: proportion, dtype: float64


## Feature Scaling

Logistic Regression requires feature scaling since input variables are on very different scales (e.g., `Platelets` in hundreds of thousands vs. `Age` in tens) — without scaling, the solver fails to converge. Random Forest and XGBoost are scale-invariant and use unscaled data.

In [ ]:
# Scale features for Logistic Regression only
scaler_a = StandardScaler()
X_train_a_scaled = scaler_a.fit_transform(X_train_a)
X_test_a_scaled = scaler_a.transform(X_test_a)

scaler_b = StandardScaler()
X_train_b_scaled = scaler_b.fit_transform(X_train_b)
X_test_b_scaled = scaler_b.transform(X_test_b)

print("Scaling applied for Logistic Regression inputs.")

Scaling applied for Logistic Regression inputs.


## Train Models — Feature Set A (Full)

In [ ]:

# Logistic Regression — Feature Set A
lr_a = LogisticRegression(max_iter=1000, random_state=42)
lr_a.fit(X_train_a_scaled, y_train_a)

# Random Forest — Feature Set A
rf_a = RandomForestClassifier(random_state=42)
rf_a.fit(X_train_a, y_train_a)

# XGBoost — Feature Set A
xgb_a = XGBClassifier(random_state=42, eval_metric="logloss")
xgb_a.fit(X_train_a, y_train_a)

print("Feature Set A: all 3 models trained.")

Feature Set A: all 3 models trained.


## Train Models — Feature Set B (Restricted)

In [ ]:

# Logistic Regression — Feature Set B
lr_b = LogisticRegression(max_iter=1000, random_state=42)
lr_b.fit(X_train_b_scaled, y_train_b)

# Random Forest — Feature Set B
rf_b = RandomForestClassifier(random_state=42)
rf_b.fit(X_train_b, y_train_b)

# XGBoost — Feature Set B
xgb_b = XGBClassifier(random_state=42, eval_metric="logloss")
xgb_b.fit(X_train_b, y_train_b)

print("Feature Set B: all 3 models trained.")

Feature Set B: all 3 models trained.


## Baseline Metrics

Full evaluation and comparison happens in Stage 06. This section captures baseline metrics to confirm training succeeded and to give an early view of results.

In [ ]:

# Evaluate a trained model on its test set and return key metrics as a dict
def evaluate(model, X_test, y_test, feature_set_name, model_name):
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    return {
        "FeatureSet": feature_set_name, "Model": model_name,
        "Accuracy": round(accuracy_score(y_test, preds), 4),
        "Precision": round(precision_score(y_test, preds), 4),
        "Recall": round(recall_score(y_test, preds), 4),
        "F1": round(f1_score(y_test, preds), 4),
        "ROC_AUC": round(roc_auc_score(y_test, proba), 4),
    }

# Evaluate all 6 model variants — Logistic Regression uses scaled test data, tree models use unscaled
results = [
    evaluate(lr_a, X_test_a_scaled, y_test_a, "A_Full", "LogisticRegression"),
    evaluate(rf_a, X_test_a, y_test_a, "A_Full", "RandomForest"),
    evaluate(xgb_a, X_test_a, y_test_a, "A_Full", "XGBoost"),
    evaluate(lr_b, X_test_b_scaled, y_test_b, "B_Restricted", "LogisticRegression"),
    evaluate(rf_b, X_test_b, y_test_b, "B_Restricted", "RandomForest"),
    evaluate(xgb_b, X_test_b, y_test_b, "B_Restricted", "XGBoost"),
]

# Save baseline metrics as a reusable results table
results_df = pd.DataFrame(results)
results_df.to_csv("MVB-05-results/MVB-05-baseline-metrics.csv", index=False)
print(results_df.to_string())

     FeatureSet               Model  Accuracy  Precision  Recall      F1  ROC_AUC
0        A_Full  LogisticRegression    0.9383     0.9268  0.9580  0.9421   0.9582
1        A_Full        RandomForest    0.9427     0.9417  0.9496  0.9456   0.9922
2        A_Full             XGBoost    0.9383     0.9412  0.9412  0.9412   0.9928
3  B_Restricted  LogisticRegression    0.9339     0.9194  0.9580  0.9383   0.9568
4  B_Restricted        RandomForest    0.9427     0.9417  0.9496  0.9456   0.9919
5  B_Restricted             XGBoost    0.9383     0.9412  0.9412  0.9412   0.9928


**Observation:** Feature Set B (Restricted, without `Pathogen_Present`) performs nearly identically to Feature Set A (Full) across all three models — differences are within a fraction of a percentage point on every metric. This confirms the Stage 03 finding that the CSF and blood markers alone already carry strong diagnostic signal: removing the near-proxy `Pathogen_Present` feature costs essentially nothing in raw performance. This is a meaningful Responsible AI result — the model does not depend on the ethically questionable proxy feature to perform well, strengthening the case for the Restricted feature set as the more defensible choice going into Stage 06.

## Save Trained Models

In [ ]:
# Bundle all 6 trained models for saving
models = {
    "MVB-05-lr-A.pkl": lr_a, "MVB-05-rf-A.pkl": rf_a, "MVB-05-xgb-A.pkl": xgb_a,
    "MVB-05-lr-B.pkl": lr_b, "MVB-05-rf-B.pkl": rf_b, "MVB-05-xgb-B.pkl": xgb_b,
}

# Save each model to the Models/ folder
for filename, model in models.items():
    with open(f"Models/{filename}", "wb") as f:
        pickle.dump(model, f)

# Save scalers too — needed to preprocess new data for Logistic Regression at inference time
with open("Models/MVB-05-scaler-A.pkl", "wb") as f:
    pickle.dump(scaler_a, f)
with open("Models/MVB-05-scaler-B.pkl", "wb") as f:
    pickle.dump(scaler_b, f)

print("All 6 models and 2 scalers saved to Models/.")

All 6 models and 2 scalers saved to Models/.


## Summary

- 6 model variants trained: Logistic Regression, Random Forest, XGBoost × Feature Set A (Full) / B (Restricted).
- 80/20 stratified train/test split, random seed 42.
- Logistic Regression required feature scaling (StandardScaler) to converge; Random Forest and XGBoost used unscaled data.
- Baseline metrics show Feature Set B performs nearly identically to Feature Set A, despite excluding the near-proxy `Pathogen_Present` feature — a meaningful Responsible AI finding.
- All 6 trained models and 2 scalers saved for use in Stage 06 (Evaluation) and Stage 07 (Explainability).

---

## Stage Status

**Stage:** Completed

**Primary Outputs Produced:**
✓ 6 trained models + 2 scalers (Models/)
✓ Baseline metrics table (MVB-05-baseline-metrics.csv)
✓ Model training notebook
✓ Updated documentation

**Input for Next Stage:**
`Models/` (6 trained models), `MVB-05-results/MVB-05-baseline-metrics.csv`

**Next Stage:**
06_MVB_Evaluation — Full evaluation including confusion matrices, false-negative analysis, and detailed metric comparison across all 6 variants.

In [14]:

# Notebook completion timestamp and environment info
from datetime import datetime
import platform
import sklearn

print("="*60)
print("Stage 05 completed successfully")
print("Completion time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Python:", platform.python_version())
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("="*60)

Stage 05 completed successfully
Completion time: 2026-07-26 08:16:36
Python: 3.12.13
Pandas: 2.2.2
Scikit-learn: 1.6.1
